In [6]:
import json
import pandas as pd
import ast
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
personas = pd.read_excel("../data/ZA9089_JSON.xlsx")

# Export a single column as JSON
personas["top-128"].to_json("../data/german_personas.json", orient="records")

In [8]:
def safe_parse(x):
    try: return ast.literal_eval(x)
    except: return eval(x)

df_expanded = personas["top-128"].apply(safe_parse).apply(pd.Series)


In [9]:
df_expanded = df_expanded.replace(r'^\(\d+\)\s*', '', regex=True)

In [ ]:
# Separate numeric and categorical columns
numeric_cols = df_expanded.select_dtypes(include="number").columns.tolist()
cat_cols = [
    c for c in df_expanded.select_dtypes(exclude="number").columns
    if df_expanded[c].nunique() <= 10
]

# --- Numeric: histogram per column ---
if numeric_cols:
    df_num = df_expanded[numeric_cols].melt(var_name="Item", value_name="Wert").dropna()
    df_num["Item"] = df_num["Item"].str[:60]
    g_num = sns.FacetGrid(df_num, col="Item", col_wrap=4, height=3, sharex=False, sharey=False)
    g_num.map(sns.histplot, "Wert", bins=20)
    g_num.set_titles("{col_name}", size=7)
    g_num.figure.suptitle("Numerische Items", y=1.01)
    plt.tight_layout()
    plt.savefig("../img/distributions_numeric.png", dpi=150, bbox_inches="tight")
    plt.show()

# --- Categorical: horizontal bar chart per column ---
if cat_cols:
    df_cat = df_expanded[cat_cols].melt(var_name="Item", value_name="Wert").dropna()
    df_cat["Item"] = df_cat["Item"].str[:60]

    g_cat = sns.FacetGrid(
        df_cat, col="Item", col_wrap=4, height=3, aspect=1.6,
        sharex=False, sharey=False,
    )
    def _countplot(data, **kwargs):
        order = data["Wert"].value_counts().index
        sns.countplot(data=data, y="Wert", order=order, **kwargs)
    g_cat.map_dataframe(_countplot)
    g_cat.set_titles("{col_name}", size=7)
    g_cat.set_axis_labels("Anzahl", "")
    g_cat.figure.suptitle("Kategoriale Items (≤10 Ausprägungen)", y=1.01)
    plt.tight_layout()
    plt.savefig("../img/distributions_categorical.png", dpi=150, bbox_inches="tight")
    plt.show()